# Online Retail — đọc, làm sạch và tạo one-hot theo hóa đơn

Notebook thực hiện đủ các bước trong đề: đọc workbook Excel, làm sạch Description và InvoiceNo, loại hóa đơn tín dụng có ký tự C, đếm số dòng theo quốc gia, gom Quantity cho hóa đơn tại United Kingdom và tạo ma trận one-hot với mỗi dòng tương ứng một hóa đơn.

Nếu chạy trên máy có D:\Online Retail.xlsx hoặc thư mục hiện tại có Online Retail.xlsx, notebook sẽ dùng tệp đó. Trên Google Colab, nếu không có tệp cục bộ, notebook sẽ tải workbook qua Google Drive ID được cung cấp.


## 1. Thư viện và đọc workbook


In [1]:
import io
import os
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display



print("Lê Quốc Trung - 6451071082")
FILE_ID = "12q4inbGbTS1UKs9sHA1X87vSRV8feEHV"
LOCAL_CANDIDATES = [
    Path("Online Retail.xlsx"),
    Path(r"D:\Online Retail.xlsx"),
    Path("/content/Online Retail.xlsx"),
]
excel_path = next((path for path in LOCAL_CANDIDATES if path.is_file()), None)


def is_xlsx(path):
    return path.is_file() and path.stat().st_size > 0 and zipfile.is_zipfile(path)


def download_shared_workbook(file_id, destination):
    errors = []

    # GDrive download, if gdown is available or can be installed.
    gdown_module = None
    try:
        import gdown as gdown_module
    except ImportError:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
            import gdown as gdown_module
        except Exception as exc:
            errors.append(f"Không cài được gdown: {exc}")

    if gdown_module is not None:
        try:
            gdown_module.download(id=file_id, output=str(destination), quiet=True, fuzzy=True)
            if is_xlsx(destination):
                return destination
            errors.append("Link Drive không trả về tệp Excel hợp lệ.")
        except Exception as exc:
            errors.append(f"Tải trực tiếp từ Drive thất bại: {exc}")

    # Dự phòng nếu link là Google Sheets thay vì tệp Excel trên Drive.
    try:
        export_url = f"https://docs.google.com/spreadsheets/d/{file_id}/export?format=xlsx"
        request = urllib.request.Request(export_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=60) as response:
            destination.write_bytes(response.read())
        if is_xlsx(destination):
            return destination
        errors.append("Google Sheets export không trả về workbook Excel hợp lệ.")
    except Exception as exc:
        errors.append(f"Google Sheets export thất bại: {exc}")

    # Dự phòng công khai cùng bộ dữ liệu chính thức UCI Online Retail.
    try:
        uci_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
        request = urllib.request.Request(uci_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=120) as response:
            destination.write_bytes(response.read())
        if is_xlsx(destination):
            print("Không tải được Google Drive; đã dùng bản Online Retail chính thức từ UCI.")
            return destination
        errors.append("Tệp tải từ UCI không phải workbook Excel hợp lệ.")
    except Exception as exc:
        errors.append(f"Tải bản dự phòng UCI thất bại: {exc}")

    destination.unlink(missing_ok=True)
    raise RuntimeError(
        "Không tải được workbook. Kiểm tra quyền chia sẻ Drive hoặc tải Online Retail.xlsx "
        "lên phiên Colab.\n" + "\n".join(errors)
    )

if excel_path is None:
    excel_path = download_shared_workbook(FILE_ID, Path("Online Retail.xlsx"))

print(f"Đọc workbook: {excel_path.resolve()}")
excel_file = pd.ExcelFile(excel_path, engine="openpyxl")
print("Các sheet:", excel_file.sheet_names)
df = pd.read_excel(excel_path, sheet_name=0, engine="openpyxl")
print("Kích thước dữ liệu ban đầu:", df.shape)
display(df.head())


Lê Quốc Trung - 6451071082
Đọc workbook: /content/Online Retail.xlsx
Các sheet: ['Online Retail']
Kích thước dữ liệu ban đầu: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 2. Làm sạch mô tả và số hóa đơn

Loại bỏ khoảng trắng/ký tự xuống dòng thừa ở đầu/cuối Description, gộp chuỗi khoảng trắng liên tiếp, bỏ các dòng không có số hóa đơn và chuyển InvoiceNo thành chuỗi để giữ đúng định danh hóa đơn.


In [2]:
print("Lê Quốc Trung - 6451071082")
required_columns = {"InvoiceNo", "Description", "Quantity", "Country"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Workbook thiếu các cột bắt buộc: {sorted(missing_columns)}")

rows_before = len(df)
df["Description"] = (
    df["Description"]
      .astype("string")
      .str.replace(r"[\r\n\t]+", " ", regex=True)
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)
df["Description"] = df["Description"].replace("", pd.NA)

df["InvoiceNo"] = df["InvoiceNo"].astype("string").str.strip()
df["InvoiceNo"] = df["InvoiceNo"].replace("", pd.NA)
missing_invoice_rows = int(df["InvoiceNo"].isna().sum())
df = df.dropna(subset=["InvoiceNo"]).copy()
df["InvoiceNo"] = df["InvoiceNo"].astype(str)

print(f"Số dòng ban đầu: {rows_before:,}")
print(f"Dòng bị xóa do thiếu InvoiceNo: {missing_invoice_rows:,}")
print("InvoiceNo sau xử lý:", df["InvoiceNo"].dtype)
print("Số giá trị InvoiceNo rỗng còn lại:", int(df["InvoiceNo"].eq("").sum()))
display(df[["InvoiceNo", "Description", "Quantity", "Country"]].head())


Lê Quốc Trung - 6451071082
Số dòng ban đầu: 541,909
Dòng bị xóa do thiếu InvoiceNo: 0
InvoiceNo sau xử lý: object
Số giá trị InvoiceNo rỗng còn lại: 0


,InvoiceNo,Description,Quantity,Country
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,United Kingdom
1,536365,WHITE METAL LANTERN,6,United Kingdom
2,536365,CREAM CUPID HEARTS COAT HANGER,8,United Kingdom
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,United Kingdom
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,United Kingdom


## 3. Loại hóa đơn tín dụng

Xóa toàn bộ dòng có ký tự C trong InvoiceNo (không phân biệt hoa/thường), theo yêu cầu đề bài.


In [3]:
print("Lê Quốc Trung - 6451071082")
credit_mask = df["InvoiceNo"].str.contains("C", case=False, regex=False, na=False)
credit_rows = int(credit_mask.sum())
credit_invoices = int(df.loc[credit_mask, "InvoiceNo"].nunique())
transactions = df.loc[~credit_mask].copy()

print(f"Số dòng thuộc hóa đơn tín dụng bị xóa: {credit_rows:,}")
print(f"Số hóa đơn tín dụng khác nhau bị xóa: {credit_invoices:,}")
print(f"Số dòng còn lại: {len(transactions):,}")
print("Còn InvoiceNo chứa ký tự C:", bool(transactions["InvoiceNo"].str.contains("C", case=False, regex=False).any()))


Lê Quốc Trung - 6451071082
Số dòng thuộc hóa đơn tín dụng bị xóa: 9,288
Số hóa đơn tín dụng khác nhau bị xóa: 3,836
Số dòng còn lại: 532,621
Còn InvoiceNo chứa ký tự C: False


## 4. Thống kê số dòng theo quốc gia


In [4]:
print("Lê Quốc Trung - 6451071082")
country_row_counts = (
    transactions.groupby("Country", dropna=False)
                .size()
                .rename("RowCount")
                .reset_index()
                .sort_values("RowCount", ascending=False)
                .reset_index(drop=True)
)
print("Số quốc gia:", len(country_row_counts))
display(country_row_counts)


Lê Quốc Trung - 6451071082
Số quốc gia: 38


,Country,RowCount
0,United Kingdom,487622
1,Germany,9042
2,France,8408
3,EIRE,7894
4,Spain,2485
5,Netherlands,2363
6,Belgium,2031
7,Switzerland,1967
8,Portugal,1501
9,Australia,1185


## 5. Lọc United Kingdom và gom Quantity theo hóa đơn, mặt hàng


In [5]:
print("Lê Quốc Trung - 6451071082")
uk_rows = transactions.loc[transactions["Country"].astype("string").str.strip().eq("United Kingdom")].copy()
uk_rows["Quantity"] = pd.to_numeric(uk_rows["Quantity"], errors="coerce")

uk_invoice_items = (
    uk_rows.groupby(["InvoiceNo", "Description"], as_index=False)["Quantity"]
           .sum()
           .sort_values(["InvoiceNo", "Description"], kind="stable")
           .reset_index(drop=True)
)

print(f"Số dòng giao dịch tại United Kingdom: {len(uk_rows):,}")
print(f"Số hóa đơn tại United Kingdom: {uk_rows['InvoiceNo'].nunique():,}")
print(f"Số cặp hóa đơn-mặt hàng sau gom nhóm: {len(uk_invoice_items):,}")
display(uk_invoice_items.head(20))


Lê Quốc Trung - 6451071082
Số dòng giao dịch tại United Kingdom: 487,622
Số hóa đơn tại United Kingdom: 20,122
Số cặp hóa đơn-mặt hàng sau gom nhóm: 475,540


,InvoiceNo,Description,Quantity
0,536365,CREAM CUPID HEARTS COAT HANGER,8
1,536365,GLASS STAR FROSTED T-LIGHT HOLDER,6
2,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6
3,536365,RED WOOLLY HOTTIE WHITE HEART.,6
4,536365,SET 7 BABUSHKA NESTING BOXES,2
5,536365,WHITE HANGING HEART T-LIGHT HOLDER,6
6,536365,WHITE METAL LANTERN,6
7,536366,HAND WARMER RED POLKA DOT,6
8,536366,HAND WARMER UNION JACK,6
9,536367,ASSORTED COLOUR BIRD ORNAMENT,32


## 6. One-hot encoding — mỗi dòng là một hóa đơn

Tạo ma trận thưa để giảm bộ nhớ: hàng là InvoiceNo, cột là Description, giá trị 1 nếu hóa đơn có mua mặt hàng đó và 0 nếu không. Chỉ tính mặt hàng có Quantity ròng lớn hơn 0 là đã mua; các mặt hàng hoàn/trả có số lượng ròng 0 hoặc âm không được đánh dấu là mua.


In [6]:
print("Lê Quốc Trung - 6451071082")
invoice_ids = pd.Index(
    uk_rows["InvoiceNo"].drop_duplicates().sort_values().to_numpy(),
    name="InvoiceNo"
)
item_ids = pd.Index(
    uk_invoice_items["Description"].dropna().drop_duplicates().sort_values().to_numpy(),
    name="Description"
)

invoice_positions = invoice_ids.get_indexer(uk_invoice_items["InvoiceNo"])
item_positions = item_ids.get_indexer(uk_invoice_items["Description"])
purchased = uk_invoice_items["Quantity"].gt(0).to_numpy()
positive_pairs = pd.DataFrame({
    "invoice_position": invoice_positions[purchased],
    "item_position": item_positions[purchased],
})
invoice_positions_by_item = {
    int(item_position): group.to_numpy(dtype=int)
    for item_position, group in positive_pairs.groupby("item_position")["invoice_position"]
}

one_hot_columns = {}
for item_position, description in enumerate(item_ids):
    values = np.zeros(len(invoice_ids), dtype=np.uint8)
    matching_invoices = invoice_positions_by_item.get(item_position)
    if matching_invoices is not None:
        values[np.unique(matching_invoices)] = 1
    one_hot_columns[description] = pd.arrays.SparseArray(values, fill_value=0)

one_hot = pd.DataFrame(one_hot_columns, index=invoice_ids)
one_hot.columns.name = "Description"

print("One-hot dimensions (invoices x products):", one_hot.shape)
print("Number of positive invoice-product pairs:", int(purchased.sum()))
print("Storage dtype:", one_hot.dtypes.iloc[0] if one_hot.shape[1] else "no products")
print("Preview (10 invoices x first 12 products):")
display(one_hot.iloc[:10, :12])

Lê Quốc Trung - 6451071082
One-hot dimensions (invoices x products): (20122, 4168)
Number of positive invoice-product pairs: 475066
Storage dtype: Sparse[uint8, 0]
Preview (10 invoices x first 12 products):


Description,*Boombox Ipod Classic,*USB Office Mirror Ball,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 DAISY PEGS IN WOOD BOX,12 EGG HOUSE PAINTED WOOD,12 HANGING EGGS HAND PAINTED,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,12 PENCILS SMALL TUBE RED RETROSPOT,12 PENCILS SMALL TUBE SKULL
InvoiceNo,,,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0,0,0
536369,0,0,0,0,0,0,0,0,0,0,0,0
536371,0,0,0,0,0,0,0,0,0,0,0,0
536372,0,0,0,0,0,0,0,0,0,0,0,0
536373,0,0,0,0,0,0,0,0,0,0,0,0
536374,0,0,0,0,0,0,0,0,0,0,0,0


## 7. Lưu kết quả


In [7]:
output_dir = Path("online_retail_outputs")
output_dir.mkdir(exist_ok=True)

# Save the cleaned transactions, country counts, grouped UK sales and sparse one-hot table.
transactions.to_csv(output_dir / "online_retail_clean.csv", index=False)
country_row_counts.to_csv(output_dir / "row_counts_by_country.csv", index=False)
uk_invoice_items.to_csv(output_dir / "uk_invoice_items_grouped.csv", index=False)
one_hot.to_pickle(output_dir / "uk_invoice_one_hot.pkl", compression="gzip")

print("Saved outputs in:", output_dir.resolve())
for path in sorted(output_dir.iterdir()):
    print(f"- {path.name}: {path.stat().st_size / (1024**2):.2f} MB")
saved_one_hot = pd.read_pickle(output_dir / "uk_invoice_one_hot.pkl", compression="gzip")
print("Reloaded one-hot shape:", saved_one_hot.shape)

Saved outputs in: /content/online_retail_outputs
- online_retail_clean.csv: 44.88 MB
- row_counts_by_country.csv: 0.00 MB
- uk_invoice_items_grouped.csv: 16.67 MB
- uk_invoice_one_hot.pkl: 0.98 MB
Reloaded one-hot shape: (20122, 4168)


## Kết luận

Notebook đã đọc Excel, làm sạch Description, loại dòng thiếu InvoiceNo, chuyển mã hóa hóa đơn thành chuỗi, xóa hóa đơn có C, thống kê giao dịch theo quốc gia, gom số lượng từng mặt hàng theo hóa đơn tại United Kingdom và tạo ma trận one-hot thưa. Các tệp kết quả có thể dùng tiếp cho phân tích giỏ hàng.
